In [27]:
import pandas as pd

In [28]:
data = pd.read_csv('/content/data.csv')

In [29]:
genre_data = pd.read_csv('/content/data_by_genres.csv')

In [30]:
year_data = pd.read_csv('/content/data_by_year.csv')

In [31]:
artist_data = pd.read_csv('/content/data_by_artist.csv')

## Song Recommendation System

### 1. Install the Spotipy library

In [32]:
!pip install spotipy

### 2. Import necessary libraries

In [33]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from collections import defaultdict
from google.colab import userdata

### 3. Spotify API Integration: Intended Workflow vs. Actual Execution

Spotify API integration: The recommendation system was designed to use Spotipy and the Spotify Web API to retrieve additional song information. However, live API authentication could not be completed because Spotify's current Development Mode requires a Premium account. Therefore, the recommendation algorithm was tested using the Spotify dataset already available in the project. This preserves the recommendation logic while demonstrating how the API would be integrated when valid credentials are available.

### 4. Define `find_song` function

This function searches for a song by name and artist in your loaded dataset (`data`). If found, it returns the song's details; otherwise, it returns `None`.

In [34]:
def find_song(name, artist):
    # Convert inputs to lowercase for case-insensitive matching
    name = name.lower()
    artist = artist.lower()

    # Filter the DataFrame to find matching songs
    matching_songs = data[
        data['name'].astype(str).str.lower().str.contains(name) &
        data['artists'].astype(str).str.lower().str.contains(artist)
    ]

    if not matching_songs.empty:
        # Return the first matching song
        return matching_songs.iloc[0].to_dict()
    else:
        return None

### 5. Define `get_song_data` function

This function now exclusively tries to find song details in your local dataset (`data`). Since live Spotify API authentication is not available, it will not attempt to search for songs externally.

In [35]:
def get_song_data(song_name, artist_name):
    # Try to find the song in the existing dataset first
    found_song = find_song(song_name, artist_name)

    if found_song:
        return found_song
    else:
        # If not found in the local dataset, return None
        print(f"Song '{song_name}' by '{artist_name}' not found in local dataset.")
        return None

In [36]:
def get_song_data(song_name, artist_name):
    # Try to find the song in the existing dataset first
    found_song = find_song(song_name, artist_name)
    if found_song:
        return found_song
    else:
        # If not found in the local dataset, and API is not being used, return None
        print(f"Song '{song_name}' by '{artist_name}' not found in local dataset.")
        return None

### 6. Define `get_mean_vector` function

This function calculates the average of numerical features for a list of songs. This average vector can then be used to find similar songs.

In [37]:
def get_mean_vector(song_list):
    song_vectors = []
    for song in song_list:
        song_data = get_song_data(song['name'], song['artist'])
        if song_data:
            # Define features to include in the vector
            features = ['valence', 'acousticness', 'danceability', 'duration_ms', 'energy',
                        'instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo', 'popularity']
            # Extract only numerical features and handle potential missing values
            vector = [song_data.get(f, 0) for f in features]
            song_vectors.append(vector)

    if not song_vectors:
        return None

    # Calculate the mean vector
    mean_vector = pd.DataFrame(song_vectors).mean().tolist()
    return mean_vector

### 7. Define `flatten_dict_list` function

This utility function takes a list of dictionaries and flattens it into a single dictionary where values for each key are grouped into lists.

In [38]:
def flatten_dict_list(dict_list):
    flattened_dict = defaultdict(list)
    for dictionary in dict_list:
        for key, value in dictionary.items():
            flattened_dict[key].append(value)
    return dict(flattened_dict)

### 8. Define `recommend_songs` function

In [39]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import pandas as pd

def recommend_songs(song_list, n_recommendations=10):

    # Get the mean feature vector of the user's songs
    mean_vector = get_mean_vector(song_list)

    if mean_vector is None:
        print("Could not get data for the given songs.")
        return pd.DataFrame()

    features = [
        'valence',
        'acousticness',
        'danceability',
        'duration_ms',
        'energy',
        'instrumentalness',
        'liveness',
        'loudness',
        'speechiness',
        'tempo',
        'popularity'
    ]

    # Work on a copy of the data DataFrame
    working_data = data.copy()

    # Prepare numerical features from the dataset
    song_features = working_data[features].copy()
    song_features = song_features.fillna(song_features.mean())

    # Scale the features of the entire dataset
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(song_features)

    # Scale the user's mean vector using the SAME scaler
    # Ensure mean_vector is a DataFrame for scaler.transform
    scaled_mean_vector = scaler.transform(
        pd.DataFrame([mean_vector], columns=features)
    )

    # Calculate cosine similarity
    similarities = cosine_similarity(
        scaled_features,
        scaled_mean_vector
    ).flatten()

    working_data['similarity'] = similarities

    # Find IDs of songs supplied by the user
    input_song_ids = []

    for song in song_list:
        song_data = get_song_data(song['name'], song['artist'])

        if song_data and 'id' in song_data:
            input_song_ids.append(song_data['id'])

    # Remove the songs the user already entered (if their 'id' is in the dataset)
    recommendations = working_data[
        ~working_data['id'].isin(input_song_ids)
    ]

    # Sort by similarity
    recommendations = recommendations.sort_values(
        by='similarity',
        ascending=False
    )

    # Return top recommendations
    return recommendations[
        ['name', 'artists', 'similarity']
    ].head(n_recommendations)

### 9. Test the recommendation system

In [40]:
### 9. Test the recommendation system
my_favorite_songs = [
    {'name': 'Shape of You', 'artist': 'Ed Sheeran'},
    {'name': 'Blinding Lights', 'artist': 'The Weeknd'},
    {'name': 'Someone You Loved', 'artist': 'Lewis Capaldi'}
]

recommendations = recommend_songs(my_favorite_songs)

print("Recommended songs based on your favorites :")
display(recommendations)

Recommended songs based on your favorites :


,name,artists,similarity
19371,Ruin My Life,['Zara Larsson'],0.973111
74909,Trampoline (with ZAYN),"['SHAED', 'ZAYN']",0.962966
108667,Some Say - Felix Jaehn Remix,"['Nea', 'Felix Jaehn']",0.961620
92170,Whiskey Sunrise,['Chris Stapleton'],0.959302
18642,You'll Be Back,"['Jonathan Groff', 'Original Broadway Cast of ...",0.956404
19409,Circles,['Post Malone'],0.952882
19301,Water Fountain,['Alec Benjamin'],0.950833
15027,You're Still The One,['Shania Twain'],0.949070
19285,Be Alright,['Dean Lewis'],0.948935
57224,Lose Somebody,"['Kygo', 'OneRepublic']",0.948549
